## Daten in CSV Umwandeln

In [1]:
# Nur die ersten n Epochen speichern für Effizienz
import os
from pathlib import Path
import pandas as pd

n = 100

# Einlesen der Datei
cols = ["date", "time", "epoch", "moteid", "temperature", "humidity", "light", "voltage"]
df = pd.read_csv(
    "../data/raw/data.txt",
    sep=r"\s+",
    names=cols,
    dtype=str,  # zuerst als string lesen, dann numerisch konvertieren
)

# Numerische Spalten sicher konvertieren (nicht-konvertierbare Werte -> NaN)
df["epoch"] = pd.to_numeric(df["epoch"], errors="coerce")
df["moteid"] = pd.to_numeric(df["moteid"], errors="coerce")
for c in ["temperature", "humidity", "light", "voltage"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Unvollständige Datensätze entfernen (kritische Felder)
df = df.dropna(subset=cols)

# --- FILTER: Nur die ersten 20 Epochen ---
df_filtered = df[df["epoch"] < n].copy()

# Export lokal für Notebook-Weiterverarbeitung
local_path = Path("../data/graph/data.csv")
local_path.parent.mkdir(parents=True, exist_ok=True)
df_filtered.to_csv(local_path, index=False)
print(f"CSV lokal gespeichert unter {local_path.resolve()}")

# Optional zusätzlich in Neo4j-Import-Verzeichnis schreiben
neo4j_import_dir = Path(os.getenv("NEO4J_IMPORT_DIR", "/neo4j-import"))
if neo4j_import_dir.exists():
    neo4j_import_dir.mkdir(parents=True, exist_ok=True)
    neo4j_path = neo4j_import_dir / "data.csv"
    df_filtered.to_csv(neo4j_path, index=False)
    print(f"CSV für Neo4j import exportiert nach {neo4j_path}")
else:
    print(f"Neo4j-Import-Verzeichnis {neo4j_import_dir} nicht gefunden (setze NEO4J_IMPORT_DIR oder mounte /neo4j-import)")

CSV lokal gespeichert unter /app/data/graph/data.csv
CSV für Neo4j import exportiert nach /neo4j-import/data.csv


## Daten für Baseline verschriftlichen

In [4]:
import csv
import json
from pathlib import Path

input_path = Path("../data/graph/data.csv")
output_path = Path("../data/base/data.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(input_path, "r", encoding="utf-8") as csv_file, \
        open(output_path, "w", encoding="utf-8") as jsonl_file:

    reader = csv.DictReader(csv_file, fieldnames=[
        "date", "time", "epoch", "moteid",
        "temperature", "humidity", "light", "voltage"
    ])

    for row in reader:
        # Werte extrahieren
        date = row["date"]
        time = row["time"]
        mote = row["moteid"]
        temp = row["temperature"]
        hum = row["humidity"]
        light = row["light"]
        volt = row["voltage"]
        epoch = row["epoch"]

        # Kurzsatz für BM25
        text = (
            f"Am {date} um {time} hat Sensor {mote} "
            f"{temp}°C Temperatur, {hum}% Luftfeuchtigkeit, "
            f"{light} Lux Licht und {volt}V Spannung gemessen."
        )

        # Haystack-kompatibles JSONL-Dokument
        doc = {
            "content": text,
            "meta": {
                "date": date,
                "time": time,
                "epoch": epoch,
                "moteid": mote,
                "temperature": temp,
                "humidity": hum,
                "light": light,
                "voltage": volt
            }
        }

        jsonl_file.write(json.dumps(doc, ensure_ascii=False) + "\n")
